<a href="https://colab.research.google.com/github/sileysa/Semester_5/blob/main/JS03/JS03_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
DATA_PATH = "wbc.csv"

# Membaca dataset Wisconsin Breast Cancer

In [4]:
df = pd.read_csv(DATA_PATH)
print("Ukuran data awal:", df.shape)
print(df.head(), "\n")

Ukuran data awal: (569, 33)
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_w

# Pisahkan variabel yang dapat digunakan & tidak dapat digunakan

In [6]:
unusable_cols = [c for c in ["id", "Unnamed: 32"] if c in df.columns]
# id          -> hanya identifier pasien, tidak punya nilai prediktif
# Unnamed: 32 -> seluruh isinya kosong (NaN), tidak mengandung informasi

df = df.drop(columns=unusable_cols)
print("Kolom yang tidak digunakan:", unusable_cols)
print("Kolom yang digunakan (%d kolom):" % df.shape[1], list(df.columns), "\n")

Kolom yang tidak digunakan: []
Kolom yang digunakan (31 kolom): ['diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst'] 



# Encoding kolom "diagnosis"

In [7]:
le = LabelEncoder()
df["diagnosis"] = le.fit_transform(df["diagnosis"])
print("Hasil encoding label:", dict(zip(le.classes_, le.transform(le.classes_))), "\n")

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

# Split data latih & uji (seperti pada Praktikum 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Data latih:", X_train.shape, "| Data uji:", X_test.shape, "\n")

Hasil encoding label: {'B': np.int64(0), 'M': np.int64(1)} 

Data latih: (455, 30) | Data uji: (114, 30) 



In [12]:
pipe = Pipeline([
    ("scaler", StandardScaler()),                      # 3. Standardisasi
    ("selector", SelectKBest(score_func=f_classif)),    # 4. Seleksi fitur
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),  # 5. Model
])

 # Standardisasi + SelectKBest + Logistic Regression via Pipeline

In [13]:
param_grid = {"selector__k": list(range(1, X.shape[1] + 1))}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

results = (
    pd.DataFrame(grid.cv_results_)[["param_selector__k", "mean_test_score", "std_test_score"]]
    .sort_values("param_selector__k")
    .reset_index(drop=True)
)
print("=== Akurasi cross-validation untuk tiap jumlah fitur (k) ===")
print(results.to_string(index=False), "\n")

best_k = grid.best_params_["selector__k"]
best_score = grid.best_score_
print(f"k dengan akurasi CV tertinggi (mentah): k={best_k}, akurasi={best_score:.4f}")

# 1 standard-error: pilih k PALING KECIL yang performanya masih
# setara secara statistik dengan k terbaik -> model lebih sederhana & hemat fitur
best_se = results.loc[results["param_selector__k"] == best_k, "std_test_score"].values[0] / np.sqrt(cv.get_n_splits())
threshold = best_score - best_se
simple_k = int(results.loc[results["mean_test_score"] >= threshold, "param_selector__k"].min())
print(f"k paling sederhana dgn performa setara (1-SE rule): k={simple_k}\n")


=== Akurasi cross-validation untuk tiap jumlah fitur (k) ===
 param_selector__k  mean_test_score  std_test_score
                 1         0.903297        0.024474
                 2         0.940659        0.027451
                 3         0.945055        0.021978
                 4         0.940659        0.026556
                 5         0.942857        0.028146
                 6         0.947253        0.032894
                 7         0.949451        0.035165
                 8         0.949451        0.035165
                 9         0.947253        0.035710
                10         0.953846        0.032151
                11         0.947253        0.027274
                12         0.947253        0.025441
                13         0.947253        0.035027
                14         0.940659        0.032301
                15         0.942857        0.032151
                16         0.945055        0.025059
                17         0.964835        0.010767
   

# Fitur-fitur yang terpilih untuk k terbaik (parsimoni)

In [14]:
scaler = StandardScaler().fit(X_train)
selector = SelectKBest(score_func=f_classif, k=simple_k).fit(scaler.transform(X_train), y_train)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
selected_features = list(scores.head(simple_k).index)

print(f"=== {simple_k} fitur terbaik yang dipilih (diurutkan berdasar skor F) ===")
for i, f in enumerate(selected_features, 1):
    print(f"{i:2d}. {f:30s}  skor F = {scores[f]:.1f}")

=== 19 fitur terbaik yang dipilih (diurutkan berdasar skor F) ===
 1. concave points_worst            skor F = 733.7
 2. perimeter_worst                 skor F = 717.2
 3. radius_worst                    skor F = 692.9
 4. concave points_mean             skor F = 684.5
 5. perimeter_mean                  skor F = 548.4
 6. area_worst                      skor F = 522.2
 7. radius_mean                     skor F = 511.3
 8. area_mean                       skor F = 444.9
 9. concavity_mean                  skor F = 397.6
10. concavity_worst                 skor F = 319.5
11. compactness_mean                skor F = 263.6
12. compactness_worst               skor F = 238.2
13. radius_se                       skor F = 205.4
14. perimeter_se                    skor F = 193.2
15. area_se                         skor F = 180.6
16. texture_worst                   skor F = 126.1
17. smoothness_worst                skor F = 103.7
18. symmetry_worst                  skor F = 100.6
19. texture_mean

# Evaluasi akhir model di data uji, menggunakan k terpilih

In [15]:
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=simple_k)),
    ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)

print("\n=== Evaluasi model akhir pada data uji ===")
print("Akurasi:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


=== Evaluasi model akhir pada data uji ===
Akurasi: 0.9824561403508771
              precision    recall  f1-score   support

           B       0.97      1.00      0.99        72
           M       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

Confusion matrix:
[[72  0]
 [ 2 40]]


Berdasarkan hasil pengujian menggunakan SelectKBest dan Logistic Regression, jumlah fitur terbaik yang digunakan adalah sebanyak 19 fitur. Fitur yang terpilih adalah concave points_worst, perimeter_worst, radius_worst, concave points_mean, perimeter_mean, area_worst, radius_mean, area_mean, concavity_mean, concavity_worst, compactness_mean, compactness_worst, radius_se, perimeter_se, area_se, texture_worst, smoothness_worst, symmetry_worst, texture_mean. Jumlah tersebut dipilih karena menghasilkan nilai akurasi terbaik dibandingkan jumlah fitur lainnya.